In [4]:
import pandas as pd
import numpy as np
import os

base_path = "../data/raw/r1/r1"

logon_file = os.path.join(base_path, "logon.csv")
device_file = os.path.join(base_path, "device.csv")
http_file = os.path.join(base_path, "http.csv")

logon = pd.read_csv(logon_file)

device = pd.read_csv(device_file)

http = pd.read_csv(
    http_file,
    header=None,
    names=["id", "date", "user", "pc", "url"]
)

print("Logon:", logon.shape)
print("Device:", device.shape)
print("HTTP:", http.shape)

Logon: (849579, 5)
Device: (65668, 5)
HTTP: (3451665, 5)


In [5]:
logon["date"] = pd.to_datetime(
    logon["date"],
    format="%m/%d/%Y %H:%M:%S"
)

device["date"] = pd.to_datetime(
    device["date"],
    format="%m/%d/%Y %H:%M:%S"
)

http["date"] = pd.to_datetime(
    http["date"],
    format="%m/%d/%Y %H:%M:%S"
)

In [6]:
logon["week"] = logon["date"].dt.to_period("W").astype(str)

logon["after_hours"] = (
    (logon["date"].dt.hour < 7) |
    (logon["date"].dt.hour >= 19)
).astype(int)

weekly_logon = logon.groupby(
    ["user", "week"]
).agg(
    total_logon_events=("activity", "count"),
    unique_pcs=("pc", "nunique"),
    logon_count=("activity", lambda x: (x == "Logon").sum()),
    logoff_count=("activity", lambda x: (x == "Logoff").sum()),
    after_hours_logon=("after_hours", "sum")
).reset_index()

In [4]:
device["week"] = device["date"].dt.to_period("W").astype(str)

device["after_hours"] = (
    (device["date"].dt.hour < 7) |
    (device["date"].dt.hour >= 19)
).astype(int)

weekly_device = device.groupby(
    ["user", "week"]
).agg(
    total_device_events=("activity", "count"),
    device_pcs=("pc", "nunique"),
    device_connect_count=(
        "activity",
        lambda x: (x == "Connect").sum()
    ),
    device_disconnect_count=(
        "activity",
        lambda x: (x == "Disconnect").sum()
    ),
    after_hours_device=("after_hours", "sum")
).reset_index()

In [7]:
weekly_http_parts = []

for chunk in pd.read_csv(
    http_file,
    header=None,
    names=["id", "date", "user", "pc", "url"],
    chunksize=100000
):

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    chunk["week"] = (
        chunk["date"]
        .dt.to_period("W")
        .astype(str)
    )

    chunk["after_hours"] = (
        (chunk["date"].dt.hour < 7) |
        (chunk["date"].dt.hour >= 19)
    ).astype(int)

    grouped = chunk.groupby(
        ["user", "week"]
    ).agg(
        http_activity_count=("url", "count"),
        after_hours_activity=("after_hours", "sum")
    ).reset_index()

    weekly_http_parts.append(grouped)

In [10]:
weekly_http = (
    pd.concat(weekly_http_parts, ignore_index=True)
    .groupby(["user", "week"])
    .agg(
        http_activity_count=("http_activity_count", "sum"),
        after_hours_activity=("after_hours_activity", "sum")
    )
    .reset_index()
)

print(weekly_http.shape)

(63509, 4)


In [11]:
weekly_features = weekly_http.merge(
    weekly_logon,
    on=["user", "week"],
    how="outer"
)

weekly_features = weekly_features.merge(
    weekly_device,
    on=["user", "week"],
    how="outer"
)

numeric_columns = [
    "http_activity_count",
    "after_hours_activity",
    "total_logon_events",
    "unique_pcs",
    "logon_count",
    "logoff_count",
    "after_hours_logon",
    "total_device_events",
    "device_pcs",
    "device_connect_count",
    "device_disconnect_count",
    "after_hours_device"
]

weekly_features[numeric_columns] = (
    weekly_features[numeric_columns]
    .fillna(0)
)

print("Shape:", weekly_features.shape)
print(weekly_features.head())

NameError: name 'weekly_device' is not defined

In [12]:
weekly_features["normal_label"] = 0

In [13]:
os.makedirs("../data/processed", exist_ok=True)

weekly_features.to_csv(
    "../data/processed/r1_weekly_normal_features.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [14]:
print("Shape:", weekly_features.shape)

Shape: (68762, 10)


Prepare a balanced prototype dataset


In [15]:
import pandas as pd

normal = pd.read_csv(
    "../data/processed/r1_weekly_normal_features.csv"
)

insider = pd.read_csv(
    "../data/processed/r42_all_scenario_features.csv"
)

print("Normal:", normal.shape)
print("Insider:", insider.shape)

Normal: (68762, 10)
Insider: (80, 17)


In [16]:
print(insider.columns.tolist())

['scenario', 'user', 'scenario_start', 'scenario_end', 'total_events', 'logon_count', 'logoff_count', 'device_connect_count', 'device_disconnect_count', 'http_count', 'after_hours_events', 'after_hours_logon', 'after_hours_http', 'wikileaks_access', 'unique_pcs', 'scenario_duration_hours', 'insider_label']


In [17]:
common_features = [
    "http_activity_count",
    "after_hours_activity",
    "total_logon_events",
    "unique_pcs",
    "logon_count",
    "logoff_count",
    "after_hours_logon",
    "total_device_events",
    "device_pcs",
    "device_connect_count",
    "device_disconnect_count",
    "after_hours_device"
]

print("Common features:")
print(common_features)

Common features:
['http_activity_count', 'after_hours_activity', 'total_logon_events', 'unique_pcs', 'logon_count', 'logoff_count', 'after_hours_logon', 'total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device']


In [18]:
normal_sample = normal.sample(
    n=700,
    random_state=42
).copy()

normal_sample["insider_label"] = 0

print(normal_sample.shape)

(700, 11)


In [19]:
insider_sample = insider.copy()

insider_sample["insider_label"] = 1

print(insider_sample.shape)

(80, 17)


In [20]:
# Make a copy
insider_sample = insider.copy()

# Rename r4.2 features to match the r1 feature names
insider_sample = insider_sample.rename(columns={
    "http_count": "http_activity_count",
    "after_hours_events": "after_hours_activity",
    "total_events": "total_logon_events",
    "device_connect_count": "device_connect_count",
    "device_disconnect_count": "device_disconnect_count",
    "after_hours_logon": "after_hours_logon",
    "unique_pcs": "unique_pcs"
})

# r4.2 does not currently have separate device_pcs,
# total_device_events, or after_hours_device.
# Derive them from the available event counts.

insider_sample["total_device_events"] = (
    insider_sample["device_connect_count"] +
    insider_sample["device_disconnect_count"]
)

insider_sample["device_pcs"] = insider_sample["unique_pcs"]

insider_sample["after_hours_device"] = 0

# Label insider scenarios
insider_sample["insider_label"] = 1

print(insider_sample.columns.tolist())

['scenario', 'user', 'scenario_start', 'scenario_end', 'total_logon_events', 'logon_count', 'logoff_count', 'device_connect_count', 'device_disconnect_count', 'http_activity_count', 'after_hours_activity', 'after_hours_logon', 'after_hours_http', 'wikileaks_access', 'unique_pcs', 'scenario_duration_hours', 'insider_label', 'total_device_events', 'device_pcs', 'after_hours_device']


In [29]:
common_features = [
    "http_activity_count",
    "after_hours_activity",
    "total_logon_events",
    "unique_pcs",
    "logon_count",
    "logoff_count",
    "after_hours_logon",
    "total_device_events",
    "device_pcs",
    "device_connect_count",
    "device_disconnect_count",
    "after_hours_device"
]

print("Number of features:", len(common_features))

Number of features: 12


In [30]:
X_normal = normal_sample[common_features].copy()

X_insider = insider_sample[common_features].copy()

y_normal = normal_sample["insider_label"]

y_insider = insider_sample["insider_label"]

X = pd.concat(
    [X_normal, X_insider],
    ignore_index=True
)

y = pd.concat(
    [y_normal, y_insider],
    ignore_index=True
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts())

KeyError: "['total_device_events', 'device_pcs', 'device_connect_count', 'device_disconnect_count', 'after_hours_device'] not in index"

In [31]:
print("Normal samples:", (y == 0).sum())
print("Insider samples:", (y == 1).sum())

print("\nInsider sample shape:")
print(insider_sample.shape)

print("\nInsider scenarios:")
print(insider_sample["scenario"].nunique())

NameError: name 'y' is not defined

In [32]:
print(X.columns.tolist())
print("\nMissing values:")
print(X.isnull().sum())

print("\nData types:")
print(X.dtypes)

NameError: name 'X' is not defined

In [33]:
print(X.describe().T)

NameError: name 'X' is not defined

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts())
print("y_test distribution:")
print(y_test.value_counts())

NameError: name 'X' is not defined

In [3]:
logon["day"] = logon["date"].dt.date
logon["hour"] = logon["date"].dt.hour

logon["is_logon"] = (
    logon["activity"] == "Logon"
).astype(int)

logon["is_logoff"] = (
    logon["activity"] == "Logoff"
).astype(int)

logon["after_hours"] = (
    (logon["hour"] < 7) |
    (logon["hour"] >= 19)
).astype(int)

daily_logon = logon.groupby(
    ["user", "day"]
).agg(
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    total_logon_events=("activity", "count"),
    after_hours_logon=("after_hours", "sum"),
    unique_pcs=("pc", "nunique")
).reset_index()

print("Daily logon features:", daily_logon.shape)
daily_logon.head()

NameError: name 'logon' is not defined

In [36]:
# ==========================================
# Create Daily Normal Dataset from r1
# ==========================================

# ----- Logon features -----
logon["day"] = logon["date"].dt.date
logon["hour"] = logon["date"].dt.hour

logon["is_logon"] = (
    logon["activity"] == "Logon"
).astype(int)

logon["is_logoff"] = (
    logon["activity"] == "Logoff"
).astype(int)

logon["after_hours"] = (
    (logon["hour"] < 7) |
    (logon["hour"] >= 19)
).astype(int)

daily_logon = logon.groupby(
    ["user", "day"]
).agg(
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    after_hours_logon=("after_hours", "sum"),
    unique_pcs=("pc", "nunique")
).reset_index()


# ----- Device features -----
device["day"] = device["date"].dt.date
device["hour"] = device["date"].dt.hour

device["is_connect"] = (
    device["activity"] == "Connect"
).astype(int)

device["is_disconnect"] = (
    device["activity"] == "Disconnect"
).astype(int)

daily_device = device.groupby(
    ["user", "day"]
).agg(
    device_connect_count=("is_connect", "sum"),
    device_disconnect_count=("is_disconnect", "sum"),
    total_device_events=("activity", "count")
).reset_index()


# ----- HTTP features -----
http_daily_parts = []

for chunk in pd.read_csv(
    http_file,
    header=None,
    names=["id", "date", "user", "pc", "url"],
    chunksize=100000
):

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        format="%m/%d/%Y %H:%M:%S"
    )

    chunk["day"] = chunk["date"].dt.date
    chunk["hour"] = chunk["date"].dt.hour

    chunk["after_hours"] = (
        (chunk["hour"] < 7) |
        (chunk["hour"] >= 19)
    ).astype(int)

    grouped = chunk.groupby(
        ["user", "day"]
    ).agg(
        http_activity_count=("url", "count"),
        after_hours_activity=("after_hours", "sum")
    ).reset_index()

    http_daily_parts.append(grouped)


http_daily = (
    pd.concat(
        http_daily_parts,
        ignore_index=True
    )
    .groupby(["user", "day"])
    .agg(
        http_activity_count=("http_activity_count", "sum"),
        after_hours_activity=("after_hours_activity", "sum")
    )
    .reset_index()
)


# ----- Merge all daily features -----
daily_normal = pd.merge(
    daily_logon,
    daily_device,
    on=["user", "day"],
    how="outer"
)

daily_normal = pd.merge(
    daily_normal,
    http_daily,
    on=["user", "day"],
    how="outer"
)

daily_normal = daily_normal.fillna(0)


# ----- Select the 9 comparable features -----
daily_common_features = [
    "http_activity_count",
    "after_hours_activity",
    "logon_count",
    "logoff_count",
    "after_hours_logon",
    "unique_pcs",
    "device_connect_count",
    "device_disconnect_count",
    "total_device_events"
]

daily_normal = daily_normal[
    ["user", "day"] + daily_common_features
].copy()

daily_normal["insider_label"] = 0


print("===================================")
print("Daily normal dataset created")
print("===================================")
print("Shape:", daily_normal.shape)
print("Features:", daily_common_features)
print("\nFirst 5 rows:")
display(daily_normal.head())

Daily normal dataset created
Shape: (334905, 12)
Features: ['http_activity_count', 'after_hours_activity', 'logon_count', 'logoff_count', 'after_hours_logon', 'unique_pcs', 'device_connect_count', 'device_disconnect_count', 'total_device_events']

First 5 rows:


,user,day,http_activity_count,after_hours_activity,logon_count,logoff_count,after_hours_logon,unique_pcs,device_connect_count,device_disconnect_count,total_device_events,insider_label
0,DTAA/AAA0371,2010-01-04,1.0,0.0,2,1,0,1,0.0,0.0,0.0,0
1,DTAA/AAA0371,2010-01-05,0.0,0.0,2,1,0,1,0.0,0.0,0.0,0
2,DTAA/AAA0371,2010-01-06,5.0,0.0,2,1,0,1,0.0,0.0,0.0,0
3,DTAA/AAA0371,2010-01-07,36.0,0.0,1,1,0,1,0.0,0.0,0.0,0
4,DTAA/AAA0371,2010-01-08,5.0,0.0,2,1,0,1,0.0,0.0,0.0,0


In [37]:
print("daily_normal shape:", daily_normal.shape)
print("daily_normal columns:")
print(daily_normal.columns.tolist())

daily_normal shape: (334905, 12)
daily_normal columns:
['user', 'day', 'http_activity_count', 'after_hours_activity', 'logon_count', 'logoff_count', 'after_hours_logon', 'unique_pcs', 'device_connect_count', 'device_disconnect_count', 'total_device_events', 'insider_label']


In [38]:
import os

os.makedirs("../data/processed", exist_ok=True)

daily_normal.to_csv(
    "../data/processed/r1_daily_normal_features.csv",
    index=False
)

print("Daily normal dataset saved successfully.")
print("../data/processed/r1_daily_normal_features.csv")

Daily normal dataset saved successfully.
../data/processed/r1_daily_normal_features.csv


In [40]:
r42_events = pd.read_csv(
    "../data/processed/r42_all_parsed_events.csv"
)

# Convert r4.2 timestamps
r42_events["timestamp"] = pd.to_datetime(
    r42_events["timestamp"],
    errors="coerce"
)

print("r4.2 events shape:", r42_events.shape)
print("Unique scenarios:", r42_events["scenario"].nunique())

print("\nMissing timestamps:")
print(r42_events["timestamp"].isna().sum())

print("\nFirst timestamps:")
print(r42_events["timestamp"].head())

r4.2 events shape: (6843, 9)
Unique scenarios: 70

Missing timestamps:
0

First timestamps:
0   2010-10-23 01:34:19
1   2010-10-23 06:18:48
2   2010-10-23 06:26:01
3   2010-10-23 06:26:48
4   2010-10-23 06:28:17
Name: timestamp, dtype: datetime64[us]


Create Daily Insider Event Features

In [ ]:
r42_events["day"] = r42_events["timestamp"].dt.date
r42_events["hour"] = r42_events["timestamp"].dt.hour

r42_events["is_http"] = (
    r42_events["event_type"] == "http"
).astype(int)

r42_events["is_logon"] = (
    (r42_events["event_type"] == "logon") &
    (r42_events["action"] == "Logon")
).astype(int)

r42_events["is_logoff"] = (
    (r42_events["event_type"] == "logon") &
    (r42_events["action"] == "Logoff")
).astype(int)

r42_events["is_connect"] = (
    (r42_events["event_type"] == "device") &
    (r42_events["action"] == "Connect")
).astype(int)

r42_events["is_disconnect"] = (
    (r42_events["event_type"] == "device") &
    (r42_events["action"] == "Disconnect")
).astype(int)

r42_events["after_hours"] = (
    (r42_events["hour"] < 7) |
    (r42_events["hour"] >= 19)
).astype(int)

r42_events["after_hours_logon_event"] = (
    (r42_events["is_logon"] == 1) &
    (r42_events["after_hours"] == 1)
).astype(int)

r42_events["after_hours_http_event"] = (
    (r42_events["is_http"] == 1) &
    (r42_events["after_hours"] == 1)
).astype(int)

print("Daily insider event features created.")

Aggregate Daily Insider Features

In [42]:
daily_insider = r42_events.groupby(
    ["scenario", "user", "day"]
).agg(
    http_activity_count=("is_http", "sum"),
    after_hours_activity=("after_hours_http_event", "sum"),
    logon_count=("is_logon", "sum"),
    logoff_count=("is_logoff", "sum"),
    after_hours_logon=("after_hours_logon_event", "sum"),
    unique_pcs=("pc", "nunique"),
    device_connect_count=("is_connect", "sum"),
    device_disconnect_count=("is_disconnect", "sum")
).reset_index()

daily_insider["total_device_events"] = (
    daily_insider["device_connect_count"] +
    daily_insider["device_disconnect_count"]
)

daily_insider["insider_label"] = 1

print("Daily insider shape:", daily_insider.shape)
print("Unique insider scenarios:", daily_insider["scenario"].nunique())

Daily insider shape: (959, 13)
Unique insider scenarios: 70


In [43]:
# ==========================================
# Prepare Daily Insider Dataset
# ==========================================

daily_insider = daily_insider[
    [
        "scenario",
        "user",
        "day",
        *daily_common_features,
        "insider_label"
    ]
].copy()

print("Daily insider shape:", daily_insider.shape)
print("Unique insider scenarios:", daily_insider["scenario"].nunique())

Daily insider shape: (959, 13)
Unique insider scenarios: 70


: Select One Day per Insider Scenario

In [44]:
# ==========================================
# Select one representative day per scenario
# ==========================================

daily_normal_sample = daily_normal.sample(
    n=70,
    random_state=42
).copy()

daily_normal_sample["insider_label"] = 0

daily_insider_sample = (
    daily_insider
    .sort_values(["scenario", "day"])
    .drop_duplicates(
        subset="scenario",
        keep="first"
    )
    .copy()
)

print("Normal samples:", len(daily_normal_sample))
print("Insider samples:", len(daily_insider_sample))
print("Unique insider scenarios:",
      daily_insider_sample["scenario"].nunique())

Normal samples: 70
Insider samples: 70
Unique insider scenarios: 70


In [45]:
# ==========================================
# Create Daily Supervised Dataset
# ==========================================

X_normal_daily = daily_normal_sample[
    daily_common_features
].copy()

X_insider_daily = daily_insider_sample[
    daily_common_features
].copy()

# Combine normal and insider samples
X_daily = pd.concat(
    [
        X_normal_daily,
        X_insider_daily
    ],
    ignore_index=True
)

# Combine labels
y_daily = pd.concat(
    [
        daily_normal_sample["insider_label"],
        daily_insider_sample["insider_label"]
    ],
    ignore_index=True
)

print("===================================")
print("Daily supervised dataset")
print("===================================")

print("X_daily shape:", X_daily.shape)
print("y_daily shape:", y_daily.shape)

print("\nClass distribution:")
print(y_daily.value_counts())

print("\nFeatures:")
print(X_daily.columns.tolist())

Daily supervised dataset
X_daily shape: (140, 9)
y_daily shape: (140,)

Class distribution:
insider_label
0    70
1    70
Name: count, dtype: int64

Features:
['http_activity_count', 'after_hours_activity', 'logon_count', 'logoff_count', 'after_hours_logon', 'unique_pcs', 'device_connect_count', 'device_disconnect_count', 'total_device_events']


In [46]:
# ==========================================
# Save Daily Supervised Dataset
# ==========================================

X_daily.to_csv(
    "../data/processed/X_daily_supervised.csv",
    index=False
)

y_daily.to_csv(
    "../data/processed/y_daily_supervised.csv",
    index=False
)

print("Daily supervised datasets saved successfully.")
print("../data/processed/X_daily_supervised.csv")
print("../data/processed/y_daily_supervised.csv")

Daily supervised datasets saved successfully.
../data/processed/X_daily_supervised.csv
../data/processed/y_daily_supervised.csv
